![Logo](../Logo.png)

Last modified: 18.10.2026

# Title: Session 2 – Reviewing AI-Written Code: What Do Microsatellite-Unstable Tumors Lose?

## Objectives

A colleague asked an AI assistant, step by step, to analyse a real proteomics dataset: 104 endometrial tumors and 49 normal tissue samples from the CPTAC study, measured by TMT mass spectrometry. The question is the one from the proteomics session, MSI-H versus MSS tumors, now in another cancer and another technology. Every code cell below was written by the AI. Every cell runs. **Six of the 15 AI-written cells contain an error**; the others are correct. Your job is to decide which is which.

**Learning Outcomes**

By the end of this assignment you will be able to:

- Read AI-generated analysis code and predict its output before running it.
- Check whether code does what the user meant, not only what the user typed.
- Use biological knowledge (mismatch repair, tumor histology, the TMT technology) to judge whether a result is plausible.
- Find, fix and explain errors that run without any error message.

## How to work

Work in pairs: ideally one partner stronger in biology, one stronger in computing. Every step has the same structure:

- **Mission:** what this step of the analysis is for.
- **What the user asked the AI:** the request, word for word.
- **The AI's code:** run it.
- **Question: Is this what the user asked for?** Go through a) to d) and write your answer in the box below it.

When you find an error, fix it in the cell, re-run that cell and continue: later steps use the results of earlier ones (use *Run all above* if needed). If a cell is correct, say how you checked.

Scoring: 1 point for each obvious error, 2 for a medium one, 3 for a hard one, counted only with a correct explanation.

## Reading material

- Dou Y, Kawaler EA, Cui Zhou D, et al. Proteogenomic characterization of endometrial carcinoma. *Cell*, 2020.
- The proteomics session notebook (MSI-H versus MSS colorectal tumors).
- Data: the CPTAC endometrial discovery cohort, as distributed in version 0.4.3 of the `cptac` Python package (also available from the NCI Proteomic Data Commons, study PDC000125).

## Prerequisites & Technical Setup

- Google Colab or Jupyter, with pandas, numpy, scipy, scikit-learn, matplotlib and statsmodels
- Internet access in the first code cell, which downloads the data (37 MB) once
- **Data tier:** Tier 1 (public, de-identified research data)

<div style="background-color: #fef9e7; padding: 15px; border-left: 5px solid #d4ac0d; border-radius: 5px;">

**Tutor note – plan and key**

| Part | Content | Time |
|---|---|---|
| 1 | Get the data | 3 min |
| 2 | Choosing the samples | 10 min |
| 3 | Preparing the values | 8 min |
| 4 | Quality control (PCA) | 5 min |
| 5 | Testing every protein, volcano plot | 12 min |
| 6 | Repair proteins | 6 min |
| 7 | MLH1 loss and mutations | 6 min |
| 8 | Tumors that keep MLH1 | 3 min |
| 9 | Proteins higher in MSI-H | 3 min |
| 10 | Findings and Failure Log | 4 min |

Total: 60 min.

**Errors (11 points):** 2.2 serous tumors kept (hard, 3) · 3.1 log of log ratios (obvious, 1) · 5.1 fold change reversed (obvious, 1) · 5.2 volcano labels on the wrong points (medium, 2) · 6.2 plot colours on the wrong group (medium, 2) · 7.1 "lost" = below 0 (medium, 2). Steps 1.1, 2.1, 2.3, 3.2, 4.1, 6.1, 7.2, 8.1 and 9.1 are correct.

**Errors carry forward.** Students should fix each error in place and re-run before moving on. If 2.2 and 3.1 are both left unfixed, the (correct) PCA cell stops with "Input X contains NaN"; point them upstream rather than at the PCA.

**Hint ladder for 2.2**, the error most pairs miss: at about minute 25, ask "Is there anything besides MSI that differs between your two groups? Look at the table in 2.1." At Part 9, ask "Why would MSI-H tumors not be more proliferative?" 

</div>

## Part 1 – The Question and the Data (3 min)

**Question.** What do microsatellite-unstable (MSI-H) endometrial tumors lose, and gain, at the protein level compared with microsatellite-stable (MSS) tumors?

About a quarter of endometrial cancers are MSI-H: their DNA mismatch-repair system has failed, so they accumulate thousands of small insertion and deletion errors. The repair system works as two pairs of proteins: MLH1 with PMS2, and MSH2 with MSH6.

### Part 1.1 – Get the data (3 min)

**Mission.** Download the data and load the three tables: proteins (relative abundance of each protein in each sample), clinical information (one row per sample) and somatic mutations (one row per mutation, labelled by patient).

**What the user asked the AI:**

> Download the CPTAC endometrial cancer data and load the protein, clinical and mutation tables. Tell me how big they are.

In [ ]:
#Run this code block
# AI-generated code
import os, urllib.request, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from sklearn.decomposition import PCA
from statsmodels.stats.multitest import multipletests

# The data ship inside version 0.4.3 of the cptac package; download it once and unpack three files
DATA = "cptac_endometrial_data"
WHEEL = ("https://files.pythonhosted.org/packages/f8/29/f5d7591c8b1722e883645c5cdc20362f2fcd5cad764981680ff977899653/"
         "cptac-0.4.3-py3-none-any.whl")
FILES = ["proteomics.cct.gz", "clinical.txt", "somatic.maf.gz"]
if not all(os.path.exists(os.path.join(DATA, f)) for f in FILES):
    os.makedirs(DATA, exist_ok=True)
    urllib.request.urlretrieve(WHEEL, "cptac-0.4.3.whl")
    with zipfile.ZipFile("cptac-0.4.3.whl") as z:
        for f in FILES:
            with z.open(f"cptac/data_endometrial/{f}") as src, open(os.path.join(DATA, f), "wb") as dst:
                dst.write(src.read())
    os.remove("cptac-0.4.3.whl")

proteins = pd.read_csv(f"{DATA}/proteomics.cct.gz", sep="\t", index_col=0)
clinical = pd.read_csv(f"{DATA}/clinical.txt", sep="\t", index_col=0, encoding="latin-1")
mutations = pd.read_csv(f"{DATA}/somatic.maf.gz", sep="\t", low_memory=False)
mutations["patient"] = mutations["Tumor_Sample_Barcode"].str.replace("_T", "", regex=False)

print(f"Proteins:  {proteins.shape[0]:,} proteins x {proteins.shape[1]} samples")
print(f"Clinical:  {clinical.shape[0]} samples x {clinical.shape[1]} columns")
print(f"Mutations: {len(mutations):,} mutations in {mutations['patient'].nunique()} patients")
print("Same samples, same order in proteins and clinical:", (proteins.columns == clinical.index).all())

#### Q 1.1.1 Is this what the user asked for?

a) **Inspect:** read the code. What does each part do?

b) **Predict:** before running, what output do you expect?

c) **Check:** run the cell. Does the output match your prediction? Does the code do what the user meant, and is the result biologically plausible?

d) **Fix:** if something is wrong, correct the cell, re-run it and explain the change. If it is correct, say how you checked.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

<b>A1.1.1)</b>

**Correct.** The cell downloads the package file once, unpacks three tables and loads them. Checks: 10,999 proteins × 153 samples, 153 clinical rows, and the samples are in the same order in both tables (`True`). The mutation table covers 103 patients, the tumors that were sequenced. Worth noticing for later: proteins and clinical data are labelled by sample (`S001`), the mutation table by patient (`C3L-00006_T`).

</div>

## Part 2 – Choosing the Samples to Compare (10 min)

Not every sample in the data is a tumor, and not every tumor belongs in the comparison.

### Part 2.1 – What samples are in the data? (3 min)

**Mission.** Before choosing groups, see what kinds of samples the study contains.

**What the user asked the AI:**

> Show me what kinds of samples are in the data: tissue type and MSI status for every sample, and the histology of the tumors.

In [ ]:
#Run this code block
# AI-generated code
print("Tissue type by MSI status (all samples):")
print(pd.crosstab(clinical["Proteomics_Tumor_Normal"], clinical["MSI_status"].fillna("not tested")))

tumors = clinical[(clinical["Proteomics_Tumor_Normal"] == "Tumor") & (clinical["Case_excluded"] == "No")]
print("\nHistology by MSI status (tumors that passed the study's quality review):")
print(pd.crosstab(tumors["Histologic_type"], tumors["MSI_status"]))

#### Q 2.1.1 Is this what the user asked for?

a) **Inspect:** read the code. What does each part do?

b) **Predict:** before running, what output do you expect?

c) **Check:** run the cell. Does the output match your prediction? Does the code do what the user meant, and is the result biologically plausible?

d) **Fix:** if something is wrong, correct the cell, re-run it and explain the change. If it is correct, say how you checked.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

<b>A2.1.1)</b>

**Correct.** It only describes the data. 49 normal-tissue samples have no MSI status (MSI is a tumor property), 104 tumors are tested, and after the study's quality review 95 remain. The key observation for the next step: **all 12 serous tumors are MSS**; MSI-H occurs only in endometrioid tumors (25 MSI-H, 58 MSS).

</div>

### Part 2.2 – Define the two groups (4 min)

**Mission.** Choose the tumors to compare and label each one MSI-H or MSS. Every later step uses these groups.

**What the user asked the AI:**

> Set up the comparison of MSI-H and MSS tumors. Use every tumor that has an MSI result, so we keep as much statistical power as possible.

In [ ]:
#Run this code block
# AI-generated code
tumors = clinical[(clinical["Proteomics_Tumor_Normal"] == "Tumor") & (clinical["Case_excluded"] == "No")]
tumors = tumors[tumors["MSI_status"].notna()]   # every tumor with an MSI result
X = proteins[tumors.index]            # proteins x tumors, columns in the same order as `tumors`
group = tumors["MSI_status"]          # indexed by sample ID, so it lines up with the columns of X
patient = tumors["Proteomics_Participant_ID"]

assert (X.columns == group.index).all()
print(group.value_counts().to_string())

#### Q 2.2.1 Is this what the user asked for?

a) **Inspect:** read the code. What does each part do?

b) **Predict:** before running, what output do you expect?

c) **Check:** run the cell. Does the output match your prediction? Does the code do what the user meant, and is the result biologically plausible?

d) **Fix:** if something is wrong, correct the cell, re-run it and explain the change. If it is correct, say how you checked.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

<b>A2.2.1)</b>

**Error (hard, 3 points): serous tumors kept in the comparison.** The AI did exactly what was typed: every tumor with an MSI result, giving 25 MSI-H against 70 MSS. But the table in 2.1 shows that the 12 extra MSS tumors are all serous. Serous carcinoma is a different disease (TP53-driven, highly proliferative), and MSI occurs almost only in endometrioid tumors, so the MSS group now differs from the MSI-H group in histology as well as in MSI status: the comparison is confounded. Nothing looks wrong in this cell; the damage appears in Part 9. With the bug, only 3 proteins are higher in MSI-H (RAP1GAP, RPL22L1, UGDH) instead of 17, and the cell-cycle markers lose significance (MKI67 q 0.034 → 0.19, CDK1 0.008 → 0.065, CCNB1 0.018 → 0.16), because the proliferative serous tumors raise the MSS average. Significant proteins in analysis 2 fall from 60 to 31, while MLH1 and PMS2 stay on top, so the analysis still looks convincing. **What the user meant:** more statistical power. **What they needed:** groups that differ only in MSI. More samples do not help if they add a confounder. Fix: keep endometrioid tumors only (25 against 58).

**Corrected code:**

<pre style="font-size:11px; white-space:pre-wrap;">tumors = clinical[(clinical[&quot;Proteomics_Tumor_Normal&quot;] == &quot;Tumor&quot;) &amp; (clinical[&quot;Case_excluded&quot;] == &quot;No&quot;)]
tumors = tumors[(tumors[&quot;Histologic_type&quot;] == &quot;Endometrioid&quot;) &amp; tumors[&quot;MSI_status&quot;].notna()]   # serous tumors are all MSS
X = proteins[tumors.index]            # proteins x tumors, columns in the same order as `tumors`
group = tumors[&quot;MSI_status&quot;]          # indexed by sample ID, so it lines up with the columns of X
patient = tumors[&quot;Proteomics_Participant_ID&quot;]

assert (X.columns == group.index).all()
print(group.value_counts().to_string())</pre>

</div>

### Part 2.3 – Are the MSI-H tumors hypermutated? (3 min)

**Mission.** A sanity check on the labels: if mismatch repair has failed, MSI-H tumors should carry many more mutations than MSS tumors.

**What the user asked the AI:**

> Check that the MSI-H tumors really are hypermutated: count each tumor's somatic mutations and compare the two groups.

In [ ]:
#Run this code block
# AI-generated code
# The mutation table is labelled by patient (C3L-00006), the proteins by sample (S001)
burden = patient.map(mutations["patient"].value_counts()).fillna(0)
print("Somatic mutations per tumor:")
print(burden.groupby(group).describe()[["count", "25%", "50%", "75%"]].round(0))

#### Q 2.3.1 Is this what the user asked for?

a) **Inspect:** read the code. What does each part do?

b) **Predict:** before running, what output do you expect?

c) **Check:** run the cell. Does the output match your prediction? Does the code do what the user meant, and is the result biologically plausible?

d) **Fix:** if something is wrong, correct the cell, re-run it and explain the change. If it is correct, say how you checked.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

<b>A2.3.1)</b>

**Correct.** Tumors are matched to the mutation table through the patient ID (`Proteomics_Participant_ID`), because the mutation table does not use sample IDs. MSI-H tumors carry a median of 529 somatic mutations against 42 in MSS tumors, more than ten times as many, as expected when mismatch repair fails. Checks: the count column shows every tumor (25 and 58), and no tumor has zero mutations, which would signal a failed match.

</div>

## Part 3 – Preparing the Values (8 min)

In the proteomics session the values were label-free intensities and the first step was a log transform. This dataset was measured with a different technology (TMT), so check what the numbers are before transforming them.

### Part 3.1 – Prepare the values for statistics (4 min)

**Mission.** Put the protein values on the scale the statistical tests expect.

**What the user asked the AI:**

> Prepare the protein values for the statistics. Log-transform them as we did in the proteomics session (log2 of value + 1).

In [ ]:
#Run this code block
# AI-generated code
values = X.values[~np.isnan(X.values)]
print(f"Median {np.median(values):.2f}, range {values.min():.1f} to {values.max():.1f}, "
      f"negative values: {100 * (values < 0).mean():.0f}%")
plt.figure(figsize=(6, 3)); plt.hist(values, bins=100, color="#4c72b0")
plt.xlabel("value"); plt.ylabel("count"); plt.title("All protein values"); plt.tight_layout(); plt.show()

X_stats = np.log2(X + 1)        # log transform, as for the intensities in the proteomics session
print(f"Missing values: {100 * X_stats.isna().values.mean():.1f}%")

#### Q 3.1.1 Is this what the user asked for?

a) **Inspect:** read the code. What does each part do?

b) **Predict:** before running, what output do you expect?

c) **Check:** run the cell. Does the output match your prediction? Does the code do what the user meant, and is the result biologically plausible?

d) **Fix:** if something is wrong, correct the cell, re-run it and explain the change. If it is correct, say how you checked.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

<b>A3.1.1)</b>

**Error (obvious, 1 point): log transform of values that are already logs.** The first line of output already answers the question: the values are centred on 0 (median −0.00) and half are negative. They are log2 ratios to a pooled reference sample in the same TMT plex, not intensities; a log transform is not needed, and `log2` of a negative number does not exist. `log2(x + 1)` turns every value below −1 into NaN (with a RuntimeWarning) and every value of exactly −1 into minus infinity. Missing values rise from 11.2% to 14.8%; in 3.2 the complete protein set halves (8,020 → 4,080) and **MLH1 drops out of both protein sets**, because its losses in MSI-H tumors are exactly the values below −1. Later, MLH1, PMS2 and EPM2AIP1 show as NaN in 6.1, and the PCA in 4.1 can stop with "Input X contains NaN" (if the error in 2.2 is also still there). The user's request was right for the label-free data of the earlier session, not for TMT. Fix: `X_stats = X`.

**Corrected code:**

<pre style="font-size:11px; white-space:pre-wrap;">values = X.values[~np.isnan(X.values)]
print(f&quot;Median {np.median(values):.2f}, range {values.min():.1f} to {values.max():.1f}, &quot;
      f&quot;negative values: {100 * (values &lt; 0).mean():.0f}%&quot;)
plt.figure(figsize=(6, 3)); plt.hist(values, bins=100, color=&quot;#4c72b0&quot;)
plt.xlabel(&quot;value&quot;); plt.ylabel(&quot;count&quot;); plt.title(&quot;All protein values&quot;); plt.tight_layout(); plt.show()

X_stats = X       # TMT values are already log2 ratios to a pooled reference: no transform
print(f&quot;Missing values: {100 * X_stats.isna().values.mean():.1f}%&quot;)</pre>

</div>

### Part 3.2 – Handle missing values (4 min)

**Mission.** Decide which proteins have enough measurements to be tested. In TMT, samples measured together in one run (a plex) share their missing values, so the missingness is technical and is not imputed here.

**What the user asked the AI:**

> Make two protein sets: analysis 1 with the proteins measured in every tumor, and analysis 2 with the proteins measured in at least 70% of the tumors of each group. Don't impute.

In [ ]:
#Run this code block
# AI-generated code
analysis1 = X_stats[X_stats.notna().all(axis=1)]

valid_share = pd.concat({g: X_stats.loc[:, group == g].notna().mean(axis=1) for g in ["MSI-H", "MSS"]}, axis=1)
analysis2 = X_stats[(valid_share >= 0.7).all(axis=1)]

print(f"All proteins: {len(X_stats):,}")
print(f"Analysis 1 (measured in every tumor): {len(analysis1):,}")
print(f"Analysis 2 (at least 70% measured in each group): {len(analysis2):,}")
print("MLH1 in analysis 1:", "MLH1" in analysis1.index, "| in analysis 2:", "MLH1" in analysis2.index)

#### Q 3.2.1 Is this what the user asked for?

a) **Inspect:** read the code. What does each part do?

b) **Predict:** before running, what output do you expect?

c) **Check:** run the cell. Does the output match your prediction? Does the code do what the user meant, and is the result biologically plausible?

d) **Fix:** if something is wrong, correct the cell, re-run it and explain the change. If it is correct, say how you checked.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

<b>A3.2.1)</b>

**Correct.** Analysis 1 keeps the 8,020 proteins measured in every tumor, analysis 2 the 9,254 measured in at least 70% of the tumors of each group, and nothing is imputed, as asked. Not imputing is right here: in TMT a protein is missing for all samples of a plex at once, so a missing value says nothing about the protein being low. Check: MLH1 is present in both sets (if it is not, look back at 3.1). Note that the filter asks for 70% in *each* group, stricter than the proteomics session's "at least one group"; without imputation that is a sensible choice, and it matches the request.

</div>

## Part 4 – Quality Control (5 min)

A quality-control look at all tumors at once, before any group comparison.

### Part 4.1 – PCA of the tumors (5 min)

**Mission.** Check whether the tumors group by TMT plex (a batch effect) or contain outliers, and whether MSI status dominates the data.

**What the user asked the AI:**

> Run a PCA of the tumors on the proteins of analysis 1, coloured once by MSI status and once by TMT plex.

In [ ]:
#Run this code block
# AI-generated code
plex = tumors["Proteomics_TMT_plex"]
Z = analysis1.T                                    # tumors as rows
Z = (Z - Z.mean()) / Z.std()
pca = PCA(n_components=2).fit(Z)
pcs = pca.transform(Z)
expl = 100 * pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for g, color in {"MSS": "#1f77b4", "MSI-H": "#d62728"}.items():
    m = (group == g).values
    axes[0].scatter(pcs[m, 0], pcs[m, 1], s=30, color=color, edgecolor="black", linewidth=0.4, label=g)
axes[0].legend(frameon=False); axes[0].set_title("Coloured by MSI status")
axes[1].scatter(pcs[:, 0], pcs[:, 1], s=30, c=plex.values, cmap="tab20", edgecolor="black", linewidth=0.4)
axes[1].set_title("Coloured by TMT plex")
for ax in axes:
    ax.set_xlabel(f"PC1 ({expl[0]:.1f}%)"); ax.set_ylabel(f"PC2 ({expl[1]:.1f}%)")
plt.tight_layout(); plt.show()

#### Q 4.1.1 Is this what the user asked for?

a) **Inspect:** read the code. What does each part do?

b) **Predict:** before running, what output do you expect?

c) **Check:** run the cell. Does the output match your prediction? Does the code do what the user meant, and is the result biologically plausible?

d) **Fix:** if something is wrong, correct the cell, re-run it and explain the change. If it is correct, say how you checked.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

<b>A4.1.1)</b>

**Correct.** Proteins are standardised across tumors (columns of `Z`), and the colours follow the same tumor order as the PCA rows. The tumors do not group by plex: the pooled reference in each plex already removes run-to-run differences. They do not separate cleanly by MSI status either, which is expected because most proteins do not differ between the groups. A few tumors sit far left on PC1; without a technical reason they should not be removed. If the cell stops with "Input X contains NaN", the cause is upstream (3.1), not here.

</div>

## Part 5 – Testing Every Protein (12 min)

The core of the analysis: one statistical test per protein, then a volcano plot.

### Part 5.1 – Test every protein (5 min)

**Mission.** For each protein, estimate how different MSI-H and MSS tumors are (log2 fold change) and how sure we can be (Welch's t-test with Benjamini–Hochberg correction). Significant: q < 0.05 and at least a 1.5-fold change (|log2 fold change| ≥ 0.58), as in the proteomics session.

**What the user asked the AI:**

> For each protein, compute the fold change of MSS versus MSI-H tumors and a Welch t-test with FDR correction, for both protein sets. Tell me how many proteins are higher and lower in MSI-H.

In [ ]:
#Run this code block
# AI-generated code
def diff_expression(data, group):
    in_msi = data.loc[:, (group == "MSI-H").values]
    in_mss = data.loc[:, (group == "MSS").values]
    t, p = ttest_ind(in_mss, in_msi, axis=1, equal_var=False, nan_policy="omit")
    res = pd.DataFrame({"log2FC": in_mss.mean(axis=1) - in_msi.mean(axis=1),   # MSS versus MSI-H
                        "pval": np.asarray(p)}, index=data.index).dropna(subset=["pval"])
    res["padj"] = multipletests(res["pval"], method="fdr_bh")[1]
    return res.sort_values("pval")

def significant(res, col="padj"):
    return res[(res[col] < 0.05) & (res["log2FC"].abs() >= 0.58)]

res1 = diff_expression(analysis1, group)
res2 = diff_expression(analysis2, group)
for name, res in [("Analysis 1", res1), ("Analysis 2", res2)]:
    s = significant(res)
    print(f"{name}: {len(res):,} tested, {len(s)} significant "
          f"({(s['log2FC'] > 0).sum()} higher in MSI-H, {(s['log2FC'] < 0).sum()} lower)")

#### Q 5.1.1 Is this what the user asked for?

a) **Inspect:** read the code. What does each part do?

b) **Predict:** before running, what output do you expect?

c) **Check:** run the cell. Does the output match your prediction? Does the code do what the user meant, and is the result biologically plausible?

d) **Fix:** if something is wrong, correct the cell, re-run it and explain the change. If it is correct, say how you checked.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

<b>A5.1.1)</b>

**Error (obvious, 1 point): fold change in the wrong direction.** The user typed "MSS versus MSI-H", and the AI subtracted in that order, so a positive fold change now means higher in **MSS**. But the summary line, the volcano axis label and every later step read positive as "higher in MSI-H". The p-values are unaffected, so the counts look plausible: "40 higher in MSI-H, 15 lower". The biology gives it away in 6.1, where MLH1 appears 2.4-fold *higher* in MSI-H tumors and PMS2 1.8-fold higher. MSI comes from losing mismatch repair, so that cannot be. Fix: MSI-H minus MSS (or keep MSS minus MSI-H and relabel everything consistently). Correct counts: 15 higher and 40 lower in analysis 1, 17 higher and 43 lower in analysis 2.

**Corrected code:**

<pre style="font-size:11px; white-space:pre-wrap;">def diff_expression(data, group):
    in_msi = data.loc[:, (group == &quot;MSI-H&quot;).values]
    in_mss = data.loc[:, (group == &quot;MSS&quot;).values]
    t, p = ttest_ind(in_msi, in_mss, axis=1, equal_var=False, nan_policy=&quot;omit&quot;)
    res = pd.DataFrame({&quot;log2FC&quot;: in_msi.mean(axis=1) - in_mss.mean(axis=1),   # positive = higher in MSI-H
                        &quot;pval&quot;: np.asarray(p)}, index=data.index).dropna(subset=[&quot;pval&quot;])
    res[&quot;padj&quot;] = multipletests(res[&quot;pval&quot;], method=&quot;fdr_bh&quot;)[1]
    return res.sort_values(&quot;pval&quot;)

def significant(res, col=&quot;padj&quot;):
    return res[(res[col] &lt; 0.05) &amp; (res[&quot;log2FC&quot;].abs() &gt;= 0.58)]

res1 = diff_expression(analysis1, group)
res2 = diff_expression(analysis2, group)
for name, res in [(&quot;Analysis 1&quot;, res1), (&quot;Analysis 2&quot;, res2)]:
    s = significant(res)
    print(f&quot;{name}: {len(res):,} tested, {len(s)} significant &quot;
          f&quot;({(s[&#x27;log2FC&#x27;] &gt; 0).sum()} higher in MSI-H, {(s[&#x27;log2FC&#x27;] &lt; 0).sum()} lower)&quot;)</pre>

</div>

### Part 5.2 – Volcano plot (7 min)

**Mission.** Show all tested proteins at once: effect size on the x-axis, significance on the y-axis, with the proteins of interest labelled.

**What the user asked the AI:**

> Draw a volcano plot of analysis 2 and label the five proteins we care about: MLH1, PMS2, EPM2AIP1, RPL22L1 and MKI67.

In [ ]:
#Run this code block
# AI-generated code
r = res2
sig = r.index.isin(significant(r).index)
plt.figure(figsize=(7, 5.5))
plt.scatter(r.loc[~sig, "log2FC"], -np.log10(r.loc[~sig, "pval"]), s=6, c="#bbbbbb", linewidths=0)
plt.scatter(r.loc[sig, "log2FC"], -np.log10(r.loc[sig, "pval"]), s=10, c="#d62728", linewidths=0)
labels = ["MLH1", "PMS2", "EPM2AIP1", "RPL22L1", "MKI67"]
for gene, (fc, p) in zip(labels, r[["log2FC", "pval"]].head(5).values):   # label the top hits
    plt.annotate(gene, (fc, -np.log10(p)), fontsize=8, xytext=(4, 2), textcoords="offset points")
plt.axvline(0, ls="--", lw=0.8, c="k")
plt.xlabel("log2 fold change (MSI-H − MSS)"); plt.ylabel("−log10 p-value")
plt.title(f"Analysis 2: {sig.sum()} significant of {len(r):,} proteins")
plt.tight_layout(); plt.show()

#### Q 5.2.1 Is this what the user asked for?

a) **Inspect:** read the code. What does each part do?

b) **Predict:** before running, what output do you expect?

c) **Check:** run the cell. Does the output match your prediction? Does the code do what the user meant, and is the result biologically plausible?

d) **Fix:** if something is wrong, correct the cell, re-run it and explain the change. If it is correct, say how you checked.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

<b>A5.2.1)</b>

**Error (medium, 2 points): labels attached to the wrong points.** The AI assumed that the five proteins the user named are the five strongest results, and attached the names, in the order typed, to the first five rows of the table. The table is sorted by p-value (RPL22L1, MLH1, EPM2AIP1, PMS2, CFI), so only EPM2AIP1 lands on its own point. "MLH1" now marks RPL22L1, the point at the top right, as if MLH1 rose 5-fold in MSI-H tumors; "PMS2" marks MLH1, "RPL22L1" marks PMS2, and "MKI67" marks CFI, a falling protein, although MKI67 rises and is only around the 160th strongest result. The plot runs without warning; knowing that MSI comes from losing MLH1 is what reveals that its label is on the wrong side. Fix: look each protein up by name, `r.loc[gene, ...]`.

**Corrected code:**

<pre style="font-size:11px; white-space:pre-wrap;">r = res2
sig = r.index.isin(significant(r).index)
plt.figure(figsize=(7, 5.5))
plt.scatter(r.loc[~sig, &quot;log2FC&quot;], -np.log10(r.loc[~sig, &quot;pval&quot;]), s=6, c=&quot;#bbbbbb&quot;, linewidths=0)
plt.scatter(r.loc[sig, &quot;log2FC&quot;], -np.log10(r.loc[sig, &quot;pval&quot;]), s=10, c=&quot;#d62728&quot;, linewidths=0)
for gene in [&quot;MLH1&quot;, &quot;PMS2&quot;, &quot;EPM2AIP1&quot;, &quot;RPL22L1&quot;, &quot;MKI67&quot;]:
    if gene in r.index:
        plt.annotate(gene, (r.loc[gene, &quot;log2FC&quot;], -np.log10(r.loc[gene, &quot;pval&quot;])), fontsize=8,
                     xytext=(4, 2), textcoords=&quot;offset points&quot;)
plt.axvline(0, ls=&quot;--&quot;, lw=0.8, c=&quot;k&quot;)
plt.xlabel(&quot;log2 fold change (MSI-H − MSS)&quot;); plt.ylabel(&quot;−log10 p-value&quot;)
plt.title(f&quot;Analysis 2: {sig.sum()} significant of {len(r):,} proteins&quot;)
plt.tight_layout(); plt.show()</pre>

</div>

## Part 6 – Which Repair Proteins Are Lost? (6 min)

A closer look at the strongest results and at the four mismatch-repair proteins.

### Part 6.1 – The strongest results (3 min)

**Mission.** Read the top of the results table, and pull out the four mismatch-repair proteins and EPM2AIP1, which shares its promoter with MLH1.

**What the user asked the AI:**

> Show me the ten strongest results with the fold change also on the natural scale, and a small table for MLH1, PMS2, MSH2, MSH6 and EPM2AIP1.

In [ ]:
#Run this code block
# AI-generated code
top = res2.head(10).copy()
top["fold_change"] = 2 ** top["log2FC"]          # e.g. 0.42 = 42% of the MSS level
print(top[["log2FC", "fold_change"]].round(2).assign(padj=top["padj"].map("{:.1e}".format)).to_string())

mmr = ["MLH1", "PMS2", "MSH2", "MSH6"]
summary = res2.reindex(mmr + ["EPM2AIP1"])[["log2FC", "padj"]].assign(fold_change=lambda d: 2 ** d["log2FC"])
summary["padj"] = summary["padj"].map("{:.1e}".format)
print("\n" + summary.round(2).to_string())

#### Q 6.1.1 Is this what the user asked for?

a) **Inspect:** read the code. What does each part do?

b) **Predict:** before running, what output do you expect?

c) **Check:** run the cell. Does the output match your prediction? Does the code do what the user meant, and is the result biologically plausible?

d) **Fix:** if something is wrong, correct the cell, re-run it and explain the change. If it is correct, say how you checked.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

<b>A6.1.1)</b>

**Correct.** Because the values are log2, the natural-scale fold change is `2 ** log2FC`. MLH1 (0.42 of the MSS level), EPM2AIP1 (0.45) and PMS2 (0.54) are among the strongest drops in the whole proteome, while MSH2 and MSH6 do not change (0.93 and 0.92, q 0.58 and 0.70). MLH1 and PMS2 work as a pair and PMS2 is unstable without MLH1, the pattern pathologists look for on MLH1/PMS2 immunohistochemistry. EPM2AIP1 shares MLH1's promoter, so the two are silenced together when it is methylated. The top rise, RPL22L1 (5.2-fold), is a known consequence of frameshift mutations in RPL22 in MSI tumors. A NaN row here means a protein was lost upstream (3.1).

</div>

### Part 6.2 – Plot the repair proteins (3 min)

**Mission.** Show the four repair proteins tumor by tumor, so the pattern behind the numbers can be seen.

**What the user asked the AI:**

> Plot the four mismatch-repair proteins for every tumor, MSS in blue and MSI-H in red, with a legend. This is the figure for the lab meeting.

In [ ]:
#Run this code block
# AI-generated code
colors, labels = ["#1f77b4", "#d62728"], ["MSS", "MSI-H"]     # MSS blue, MSI-H red
fig, ax = plt.subplots(figsize=(7, 3.5))
for i, gene in enumerate(mmr):
    for j, g in enumerate(group.unique()):
        v = X.loc[gene, group == g].dropna()
        ax.scatter(np.full(len(v), i + (j - 0.5) * 0.3) + np.random.default_rng(i).normal(0, 0.03, len(v)), v,
                   s=10, color=colors[j], label=labels[j] if i == 0 else None)
ax.set_xticks(range(len(mmr))); ax.set_xticklabels(mmr); ax.axhline(0, c="k", lw=0.6)
ax.set_ylabel("log2 ratio to reference"); ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.tight_layout(); plt.show()

#### Q 6.2.1 Is this what the user asked for?

a) **Inspect:** read the code. What does each part do?

b) **Predict:** before running, what output do you expect?

c) **Check:** run the cell. Does the output match your prediction? Does the code do what the user meant, and is the result biologically plausible?

d) **Fix:** if something is wrong, correct the cell, re-run it and explain the change. If it is correct, say how you checked.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

<b>A6.2.1)</b>

**Error (medium, 2 points): colours and labels attached to the wrong group.** `group.unique()` returns the groups in the order they first appear, and the first tumor (S001) is MSI-H. So in the loop the MSI-H tumors get the first colour (blue) and the first label ("MSS"), and the MSS tumors are drawn in red as "MSI-H". The figure shows MSS tumors losing MLH1 and PMS2: the opposite of the table in 6.1 and of the biology. A second clue: the red "MSI-H" cloud is the larger one (58 points), but there are only 25 MSI-H tumors. Fix: take colour and label from the group name, e.g. `{"MSS": blue, "MSI-H": red}.items()`.

**Corrected code:**

<pre style="font-size:11px; white-space:pre-wrap;">fig, ax = plt.subplots(figsize=(7, 3.5))
for i, gene in enumerate(mmr):
    for j, (g, color) in enumerate({&quot;MSS&quot;: &quot;#1f77b4&quot;, &quot;MSI-H&quot;: &quot;#d62728&quot;}.items()):
        v = X.loc[gene, group == g].dropna()
        ax.scatter(np.full(len(v), i + (j - 0.5) * 0.3) + np.random.default_rng(i).normal(0, 0.03, len(v)), v,
                   s=10, color=color, label=g if i == 0 else None)
ax.set_xticks(range(len(mmr))); ax.set_xticklabels(mmr); ax.axhline(0, c=&quot;k&quot;, lw=0.6)
ax.set_ylabel(&quot;log2 ratio to reference&quot;); ax.legend(frameon=False, loc=&quot;center left&quot;, bbox_to_anchor=(1.01, 0.5))
plt.tight_layout(); plt.show()</pre>

</div>

## Part 7 – How Many Tumors Lost MLH1, and Is It Mutated? (6 min)

"Lost" needs a definition, and a loss can come from a mutation or from silencing of the gene.

### Part 7.1 – Count the tumors that lost MLH1 (3 min)

**Mission.** Turn the MLH1 values into a yes/no call per tumor and count the losses in each group.

**What the user asked the AI:**

> Count how many tumors in each group have lost MLH1 protein.

In [ ]:
#Run this code block
# AI-generated code
mlh1 = X.loc["MLH1"]
threshold = 0                                   # below the reference = lost
mlh1_lost = mlh1 < threshold

print(f"Threshold: log2 ratio {threshold:.2f}")
print("Tumors with MLH1 loss:", mlh1_lost.groupby(group).sum().to_dict(), "of", group.value_counts().to_dict())

#### Q 7.1.1 Is this what the user asked for?

a) **Inspect:** read the code. What does each part do?

b) **Predict:** before running, what output do you expect?

c) **Check:** run the cell. Does the output match your prediction? Does the code do what the user meant, and is the result biologically plausible?

d) **Fix:** if something is wrong, correct the cell, re-run it and explain the change. If it is correct, say how you checked.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

<b>A7.1.1)</b>

**Error (medium, 2 points): "lost" defined as below the reference.** A log2 ratio of 0 is the level of the pooled reference sample, not the absence of the protein: about half of all tumors fall below 0 for any protein. The MSI-H count barely changes (24 of 25), which looks like confirmation, but **12 of the 58 MSS tumors are now "MLH1-lost"**. That cannot be: a tumor without MLH1 has no working mismatch repair and would be MSI-H. The MSS count is the warning sign. Fix: a threshold anchored in the data, for example below the lowest MSS value (−0.58): 23 of 25 MSI-H tumors and 0 of 58 MSS tumors. Any cutoff is acceptable if the students justify it and check it against the MSS group.

**Corrected code:**

<pre style="font-size:11px; white-space:pre-wrap;">mlh1 = X.loc[&quot;MLH1&quot;]
threshold = mlh1[group == &quot;MSS&quot;].min()          # lost = below every MSS tumor
mlh1_lost = mlh1 &lt; threshold

print(f&quot;Threshold: log2 ratio {threshold:.2f}&quot;)
print(&quot;Tumors with MLH1 loss:&quot;, mlh1_lost.groupby(group).sum().to_dict(), &quot;of&quot;, group.value_counts().to_dict())</pre>

</div>

### Part 7.2 – Protein and mutation, patient by patient (3 min)

**Mission.** Set the MLH1 protein call next to the MLH1 mutation status for every MSI-H patient.

**What the user asked the AI:**

> Make a table of the MSI-H patients with their MLH1 and PMS2 protein, whether MLH1 is lost, and whether MLH1 is mutated.

In [ ]:
#Run this code block
# AI-generated code
tumor_proteins = X.rename(columns=patient)                 # tumors only, now labelled by patient
msi_patients = patient[group == "MSI-H"]
mlh1_mutated = set(mutations.loc[(mutations["Hugo_Symbol"] == "MLH1")
                                 & (mutations["Variant_Classification"] != "Silent"), "patient"])

per_patient = tumor_proteins.loc[["MLH1", "PMS2"], msi_patients].T
per_patient["MLH1_lost"] = per_patient["MLH1"] < threshold
per_patient["MLH1_mutated"] = per_patient.index.isin(mlh1_mutated)

print(f"MSI-H patients: {len(per_patient)}, MLH1 lost: {per_patient['MLH1_lost'].sum()}, "
      f"MLH1 mutated: {per_patient['MLH1_mutated'].sum()}")
per_patient.sort_values("MLH1").round(2)

#### Q 7.2.1 Is this what the user asked for?

a) **Inspect:** read the code. What does each part do?

b) **Predict:** before running, what output do you expect?

c) **Check:** run the cell. Does the output match your prediction? Does the code do what the user meant, and is the result biologically plausible?

d) **Fix:** if something is wrong, correct the cell, re-run it and explain the change. If it is correct, say how you checked.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

<b>A7.2.1)</b>

**Correct.** Only the tumor columns are renamed to patient IDs, so the table has exactly the 25 MSI-H patients; mutations are matched on patient, and silent mutations are excluded. 23 of 25 MSI-H tumors have lost MLH1 protein, but only 3 carry an MLH1 mutation (all three among the losses). In the other 20, MLH1 is switched off without a mutation, most likely by promoter methylation, which fits the loss of EPM2AIP1 in 6.1. (If the table says 24 lost, the threshold in 7.1 is still wrong.)

</div>

## Part 8 – The MSI-H Tumors That Keep MLH1 (3 min)

Some MSI-H tumors may have kept MLH1. If so, something else must have broken their mismatch repair.

### Part 8.1 – What else did they lose? (3 min)

**Mission.** Look at the repair proteins and repair-gene mutations of the two MSI-H tumors with the highest MLH1.

**What the user asked the AI:**

> Find the two MSI-H tumors with the highest MLH1 and show their four repair proteins next to the typical MSS tumor, with any mutations in repair genes.

In [ ]:
#Run this code block
# AI-generated code
kept = mlh1[group == "MSI-H"].nlargest(2).index
view = X.loc[mmr, kept].copy()
view["MSS median"] = X.loc[mmr, group == "MSS"].median(axis=1)
print(view.round(2).to_string())

repair_mutations = mutations[mutations["Hugo_Symbol"].isin(mmr) & (mutations["Variant_Classification"] != "Silent")]
print("\nRepair-gene mutations in these tumors:")
print(repair_mutations.loc[repair_mutations["patient"].isin(patient[kept]),
                           ["patient", "Hugo_Symbol", "Variant_Classification"]].to_string(index=False))

#### Q 8.1.1 Is this what the user asked for?

a) **Inspect:** read the code. What does each part do?

b) **Predict:** before running, what output do you expect?

c) **Check:** run the cell. Does the output match your prediction? Does the code do what the user meant, and is the result biologically plausible?

d) **Fix:** if something is wrong, correct the cell, re-run it and explain the change. If it is correct, say how you checked.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

<b>A8.1.1)</b>

**Correct.** The selection is restricted to MSI-H tumors before taking the two highest MLH1 values (S001 and S057). Both keep MLH1 (0.38 and −0.02, MSS median 0.24) but have lost MSH6 (−1.95 and −1.44, MSS median −0.03), with MSH2 lower as well (−0.77 and −0.66): the other repair pair has failed. S057 (patient C3L-01744) carries an MSH6 missense mutation. So every MSI-H tumor in the cohort has lost one of the two repair pairs.

</div>

## Part 9 – What Rises in MSI-H Tumors? (3 min)

The other side of the volcano: proteins that are higher in MSI-H tumors.

### Part 9.1 – Proteins higher in MSI-H tumors (3 min)

**Mission.** List the significant proteins of analysis 2 by direction, ready for pathway analysis in Enrichr.

**What the user asked the AI:**

> List the significant proteins of analysis 2 that are higher and lower in MSI-H, so I can paste them into Enrichr, and show three markers of dividing cells.

In [ ]:
#Run this code block
# AI-generated code
s = significant(res2)
higher_in_msi = sorted(s.index[s["log2FC"] > 0])
lower_in_msi = sorted(s.index[s["log2FC"] < 0])
print(f"Higher in MSI-H ({len(higher_in_msi)}): " + ", ".join(higher_in_msi))
print(f"\nLower in MSI-H ({len(lower_in_msi)}): " + ", ".join(lower_in_msi))

print("\nThree markers of dividing cells:")
print(res2.reindex(["MKI67", "CDK1", "CCNB1"])[["log2FC", "padj"]].round(3).to_string())

#### Q 9.1.1 Is this what the user asked for?

a) **Inspect:** read the code. What does each part do?

b) **Predict:** before running, what output do you expect?

c) **Check:** run the cell. Does the output match your prediction? Does the code do what the user meant, and is the result biologically plausible?

d) **Fix:** if something is wrong, correct the cell, re-run it and explain the change. If it is correct, say how you checked.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

<b>A9.1.1)</b>

**Correct.** The split uses the sign of the log2 fold change (right once 5.1 is fixed) and the q < 0.05, 1.5-fold rule. The 17 proteins higher in MSI-H are mostly cell-cycle and DNA-replication proteins (MKI67, CDK1, CCNB1, CCNA2, MCM4, MCM7, TYMS, RRM2), plus RPL22L1 and RAP1GAP; MKI67 (q 0.034), CDK1 (0.008) and CCNB1 (0.018) are all significant. MSI-H endometrioid tumors are more proliferative than MSS ones. The 43 lower proteins include the repair proteins and many stromal and extracellular-matrix proteins. If only 3 proteins are higher (RAP1GAP, RPL22L1, UGDH) and the markers are not significant, the serous tumors from 2.2 are still in the MSS group.

</div>

## Part 10 – Your Findings (4 min)

Run the whole notebook again from the top with all your fixes in place.

#### Q 10.1 What did the analysis find?

In four sentences: which repair proteins are lost in MSI-H tumors, how often MLH1 is lost and how often it is mutated, what the MSI-H tumors that keep MLH1 have lost instead, and which proteins are higher in MSI-H tumors.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

<b>A10.1)</b>

a. Out of about 9,000 proteins, MLH1, its partner PMS2 and EPM2AIP1 (which shares MLH1's promoter) are among the strongest drops in MSI-H tumors, while MSH2 and MSH6 do not change.

b. 23 of the 25 MSI-H tumors have lost MLH1 protein, but only 3 have an MLH1 mutation; most losses are probably due to promoter methylation.

c. The two MSI-H tumors that keep MLH1 have lost MSH6 instead (one carries an MSH6 mutation).

d. The proteins higher in MSI-H tumors are mostly cell-cycle proteins (MKI67, CDK1, CCNB1).

</div>

#### Q 10.2 Which error would have misled you most, and why?

Think of the one that changed a conclusion without any warning sign in the output. For your Failure Log: what would you now ask or check before trusting an AI-written analysis?

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

<b>A10.2)</b>

Open answer. Most pairs choose 2.2 (serous tumors): it produced no error, no warning and no odd number in its own cell, kept the headline result (MLH1 and PMS2 on top) and silently removed a finding three Parts later. Good checks to name for the Failure Log: compare the groups on everything except the variable of interest before testing; check the scale and units of the data before transforming; check that every count matches the known number of samples; look a named result up by name instead of trusting position; and ask, at each result, whether it is biologically possible (MLH1 cannot be higher in MSI-H tumors, an MSS tumor cannot lack MLH1).

</div>

## Glossary

| **Term** | **Definition** |
|---|---|
| **MSI-H / MSS** | Microsatellite instability-high / microsatellite stable: tumors with and without a failed DNA mismatch-repair system. |
| **Mismatch repair (MMR)** | The system that corrects DNA replication errors; it works as two protein pairs, MLH1–PMS2 and MSH2–MSH6. |
| **Lynch syndrome** | Inherited predisposition to colorectal, endometrial and other cancers caused by a germline mutation in a mismatch-repair gene. |
| **Promoter methylation** | Chemical silencing of a gene's promoter that switches the gene off without changing its sequence; the usual cause of MLH1 loss in sporadic MSI tumors. |
| **Endometrioid / serous carcinoma** | The two main histological types of endometrial cancer; MSI occurs almost only in endometrioid tumors, while serous tumors are TP53-driven. |
| **TMT (tandem mass tag)** | A labelling method that measures several samples together in one mass-spectrometry run (a plex); each value is a log2 ratio to a pooled reference sample in the same plex. |
| **log2 ratio** | log2(sample / reference): 0 = same as the reference, −1 = half, +1 = double. |
| **Log2 fold change** | The difference in mean log2 value between two groups; positive means higher in the first group. |
| **FDR / q-value** | False discovery rate: the expected share of false positives among the proteins called significant (Benjamini–Hochberg). |
| **PCA** | Principal component analysis: summarises thousands of proteins into a few axes of greatest variation, used here for quality control. |
| **Volcano plot** | Scatter plot of log2 fold change (x) against −log10 p-value (y) for every protein tested. |

In [ ]:
from IPython.display import display, Javascript
display(Javascript("$('div.jp-Cell-inputWrapper').hide()"))

import sys
sys.path.append("../../student_version")
import ai_tracker

ai_tracker.display_AI_decleration()